# ДЗ 2. Поиск утечек данных

Предсказываем опоздание заказа в момент его оформления.
Таргет: late_delivery, где 1 — опоздал, 0 — доставлен вовремя.

Нужно найти три утечки, объяснить их происхождение и сравнить
ROC-AUC до и после удаления.

Во всех экспериментах:
- HistGradientBoostingClassifier(random_state=42), остальные параметры по умолчанию;
- одно случайное разбиение 75/25 со стратификацией;
- ROC-AUC по вероятностям класса 1;
- без подбора параметров и создания новых признаков.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

df = pd.read_csv(
    "data/olist_delivery_export.csv",
    parse_dates=["ordered_at"]
)

category_columns = [
    "client_state",
    "partner_state",
    "category",
    "pay_method"
]

for column in category_columns:
    df[column] = df[column].astype("category")

print("Размер:", df.shape)
display(df.head())
display(df.isna().sum())
display(df["late_delivery"].value_counts())

Размер: (96470, 29)


,ordered_at,client_key,partner_id,client_state,partner_state,intra_state,dist_km,promised_days,dispatch_deadline_days,items_qty,...,installments,paid_value,late_delivery,order_hour,order_weekday,photos_qty,title_len,handover_lag_days,seller_score,record_id
0,2018-03-03 17:39:37,568843f1b07989b16ad67462e4f7eddf,0bae85eb84b9fb3bd773911e89288d54,SC,SP,0,82.424278,11.264155,5.007500,1,...,1.0,49.17,0,17,5,4.0,47.0,1.977211,0.9923,125097
1,2018-07-15 22:36:39,7fe36cf7ee4c8a68c57dbf47d31c952c,87d3c3aeb3ead335511b3ce315eb341e,PI,SP,0,2093.948450,16.057882,4.009479,1,...,1.0,127.46,0,22,6,1.0,28.0,0.604410,0.9333,108873
2,2017-11-25 00:55:18,3e7688aca4163064b392dba8b3184807,e5a3438891c0bfdb9394643f95273d8e,SP,SP,1,166.125935,15.961597,9.471771,1,...,1.0,35.41,0,0,5,NaN,NaN,9.846748,0.8122,124268
3,2017-05-21 23:24:21,9bab4772c790be2269300fe0d3a0ed69,5cf13accae3222c70a9cac40818ae839,RJ,MG,0,568.116589,22.024757,7.170949,1,...,1.0,95.01,0,23,6,2.0,51.0,1.482477,0.9660,139775
4,2018-08-06 09:09:46,839a04051a2c347397fd425d01fc3eb2,1fe5540d7c1c37a595fefbacd5570d9e,PR,RJ,0,1204.797590,35.618218,8.010266,2,...,2.0,378.38,0,9,0,1.0,51.0,1.205718,0.9500,130113


ordered_at                   0
client_key                   0
partner_id                   0
client_state                 0
partner_state                0
intra_state                  0
dist_km                    476
promised_days                0
dispatch_deadline_days       0
items_qty                    0
partners_qty                 0
skus_qty                     0
goods_value                  0
freight_value                0
freight_ratio                0
category                  1358
weight_g                    16
volume_cm3                  16
pay_method                   1
installments                 1
paid_value                   1
late_delivery                0
order_hour                   0
order_weekday                0
photos_qty                1358
title_len                 1358
handover_lag_days            1
seller_score                 0
record_id                    0
dtype: int64

late_delivery
0    88644
1     7826
Name: count, dtype: int64

In [2]:
excluded_columns = [
    "ordered_at",
    "client_key",
    "partner_id",
    "late_delivery"
]

X = df.drop(columns=excluded_columns)
y = df["late_delivery"]

all_features = X.columns.tolist()

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

display(pd.DataFrame({
    "rows": [len(train_idx), len(test_idx)],
    "late_share": [
        y.loc[train_idx].mean(),
        y.loc[test_idx].mean()
    ]
}, index=["train", "test"]))

print("Признаков:", len(all_features))

,rows,late_share
train,72352,0.081117
test,24118,0.081143


Признаков: 25


In [3]:
def evaluate_model(columns):
    model = HistGradientBoostingClassifier(random_state=SEED)

    model.fit(
        X.loc[train_idx, columns],
        y.loc[train_idx]
    )

    probabilities = model.predict_proba(
        X.loc[test_idx, columns]
    )[:, 1]

    return roc_auc_score(y.loc[test_idx], probabilities)


baseline_auc = evaluate_model(all_features)
print("Исходный ROC-AUC:", round(baseline_auc, 6))

Исходный ROC-AUC: 0.911209


## Проверка отдельных признаков

Для каждого числового признака вычисляю ROC-AUC, используя его значения
как оценки риска опоздания - без обучения отдельной модели.

Строки с пропусками исключаем только из этой проверки.
Исходные данные и разбиение train/test не меняем.


In [4]:
feature_results = []

for column in df.select_dtypes(include="number").columns:
    if column == "late_delivery":
        continue

    available = df[column].notna()

    auc = roc_auc_score(
        y.loc[available],
        df.loc[available, column]
    )

    feature_results.append({
        "column": column,
        "n_valid": int(available.sum()),
        "missing": int((~available).sum()),
        "roc_auc": auc,
        "auc_any_direction": max(auc, 1 - auc)
    })

feature_auc = pd.DataFrame(feature_results)

display(
    feature_auc
    .sort_values("auc_any_direction", ascending=False)
    .round(6)
)

,column,n_valid,missing,roc_auc,auc_any_direction
20,record_id,96470,0,0.799514,0.799514
19,seller_score,96470,0,0.298686,0.701314
18,handover_lag_days,96469,1,0.648531,0.648531
1,dist_km,95994,476,0.562776,0.562776
2,promised_days,96470,0,0.450509,0.549491
0,intra_state,96470,0,0.450552,0.549448
8,freight_value,96470,0,0.546375,0.546375
13,paid_value,96469,1,0.526051,0.526051
7,goods_value,96470,0,0.521316,0.521316
10,weight_g,96454,16,0.518712,0.518712


## Аудит столбцов

Проверяю каждый столбец по смыслу: что он означает и известно ли его
значение в момент оформления заказа. Для обычных полей предполагаю,
что выгрузка сохранила исходные значения на момент заказа. Это нужно
подтвердить по документации источника.

| Столбец | Что означает | Когда известен | Вердикт |
|---|---|---|---|
| ordered_at | Время оформления | При оформлении | Исключён по условию; задаёт момент прогноза |
| client_key | Идентификатор клиента | При оформлении | Исключён по условию |
| partner_id | Идентификатор продавца | При оформлении | Исключён по условию |
| client_state | Штат клиента | При указании адреса | Оставить исходный адрес |
| partner_state | Штат продавца | При оформлении | Оставить, если значение соответствует моменту заказа |
| intra_state | Совпадают ли штаты | Когда известны адреса | Оставить, если рассчитан по исходным адресам |
| dist_km | Расстояние между сторонами заказа | По адресам | Оставить, если это не фактический маршрут после доставки |
| promised_days | Обещанный срок доставки | Когда дано обещание | Оставить исходное обещание; проверить поздние изменения |
| dispatch_deadline_days | Плановый срок передачи | Когда установлен дедлайн | Оставить условно; проверить доступность при оформлении |
| items_qty | Количество позиций | При оформлении | Оставить исходный состав |
| partners_qty | Количество продавцов | При оформлении | Оставить исходный состав |
| skus_qty | Количество разных товаров | При оформлении | Оставить исходный состав |
| goods_value | Стоимость товаров | При оформлении | Оставить исходную сумму |
| freight_value | Стоимость доставки | При оформлении | Оставить исходный тариф |
| freight_ratio | Доля стоимости доставки | Из исходных сумм | Оставить, если рассчитана из доступных на cutoff сумм |
| category | Категория товара | По каталогу | Оставить снимок каталога на момент заказа |
| weight_g | Вес товара | По каталогу | Оставить снимок на момент заказа |
| volume_cm3 | Объём товара | По каталогу | Оставить снимок на момент заказа |
| pay_method | Способ оплаты | При выборе оплаты | Оставить условно; проверить поздние изменения |
| installments | Количество платежей | При выборе условий | Оставить исходные условия оплаты |
| paid_value | Сумма оплаты | Точный момент неизвестен | Проверить: фактическая сумма после оформления может быть недоступна |
| late_delivery | Факт опоздания | После доставки | Таргет; исключён из признаков |
| order_hour | Час оформления | При оформлении | Оставить, если получен из исходной даты |
| order_weekday | День недели оформления | При оформлении | Оставить, если получен из исходной даты |
| photos_qty | Число фотографий товара | По каталогу | Оставить снимок на момент заказа |
| title_len | Длина названия товара | По каталогу | Оставить снимок на момент заказа |
| handover_lag_days | Фактическое время до передачи перевозчику | После оформления | Утечка будущего события; удалить |
| seller_score | Показатель своевременности продавца | В выгрузке рассчитан по всем исходам | Утечка через агрегат таргета; удалить или пересчитать |
| record_id | Технический номер записи | Способ формирования номера выгрузки неизвестен | Диапазон связан с исходом; исключить и проверить источник |

In [5]:
for column in category_columns:
    print(f"\nСтолбец: {column}")

    category_stats = (
        df.groupby(column, observed=True, dropna=False)["late_delivery"]
        .agg(
            orders="size",
            late_share="mean",
        )
        .sort_values("orders", ascending=False)
    )

    with pd.option_context("display.max_rows", None):
        display(category_stats.round(4))


Столбец: client_state


,orders,late_share
client_state,,
SP,40494,0.0589
RJ,12350,0.1347
MG,11354,0.0561
RS,5344,0.0715
PR,4923,0.0500
SC,3546,0.0976
BA,3256,0.1404
DF,2080,0.0707
ES,1995,0.1223



Столбец: partner_state


,orders,late_share
partner_state,,
SP,68386,0.0877
MG,7661,0.0563
PR,7444,0.0652
RJ,4183,0.0849
SC,3558,0.0604
RS,1945,0.0432
DF,806,0.0608
BA,547,0.0585
GO,448,0.0402



Столбец: category


,orders,late_share
category,,
cama_mesa_banho,9184,0.0881
beleza_saude,8613,0.0899
esporte_lazer,7492,0.0778
informatica_acessorios,6511,0.0773
moveis_decoracao,6188,0.0861
utilidades_domesticas,5680,0.0702
relogios_presentes,5476,0.0855
telefonia,4073,0.0854
automotivo,3792,0.0865



Столбец: pay_method


,orders,late_share
pay_method,,
credit_card,72818,0.0796
boleto,19191,0.0888
voucher,2977,0.0702
debit_card,1483,0.0796
NaN,1,1.0000


Для категориальных признаков я сравнил количество заказов и долю
опозданий в каждой группе. Например, у клиентов из SP доля опозданий
около 5.89%, а из RJ — около 13.47%.

Различие долей само по себе не доказывает утечку: важно проверить,
когда признак становится известен и как он рассчитан.

Эта группировка сделана по всей выгрузке как диагностическая проверка;
она не является независимой оценкой качества модели.

In [6]:
display(
    df.groupby("late_delivery")["handover_lag_days"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("Пропусков:", df["handover_lag_days"].isna().sum())
print("Отрицательных значений:", df["handover_lag_days"].lt(0).sum())

,count,min,median,mean,max
late_delivery,,,,,
0,88644,-171.212419,2.138270,2.997441,55.101030
1,7825,-0.035197,3.437419,5.833761,125.775521


Пропусков: 1
Отрицательных значений: 165


## Утечка 1: handover_lag_days

По смыслу это фактический интервал до передачи перевозчику.
В момент оформления будущая передача ещё не произошла, поэтому её
фактическое время неизвестно.

Признак сообщает модели информацию о дальнейшем ходе доставки.
Удаляем его. Вместо него можно было бы использовать заранее известный
плановый срок или статистику ранее завершённых заказов.

В выгрузке обнаружено 165 отрицательных значений handover_lag_days,
минимум — около −171 дня. Если поле действительно означает время
от оформления до передачи перевозчику, такие значения выглядят
некорректными: передача получается раньше оформления.

Возможные причины — ошибка при объединении таблиц, неверно выбранные
даты или проблема формирования выгрузки. По одному столбцу нельзя
установить точную причину: для этого нужны исходные даты и описание
их обработки.

Это отдельная проблема качества данных. Даже после исправления ошибок
фактический срок передачи нельзя использовать при прогнозе в момент
оформления, потому что передача ещё не произошла

In [7]:
id_group = df["record_id"].ge(500000).map({
    False: "Нижний диапазон",
    True: "Верхний диапазон"
})

display(
    df.groupby(id_group)["record_id"]
    .agg(["min", "max", "count"])
)

display(pd.crosstab(id_group, y))

display(
    pd.crosstab(id_group, y, normalize="index").round(6)
)

id_inside_range = df["record_id"] % 100000

print(
    "AUC исходного ID:",
    round(roc_auc_score(y, df["record_id"]), 6)
)

print(
    "AUC номера без префикса:",
    round(roc_auc_score(y, id_inside_range), 6)
)

,min,max,count
record_id,,,
Верхний диапазон,900000,949997,23910
Нижний диапазон,100000,149996,72560


late_delivery,0,1
record_id,,
Верхний диапазон,17649,6261
Нижний диапазон,70995,1565


late_delivery,0,1
record_id,,
Верхний диапазон,0.738143,0.261857
Нижний диапазон,0.978432,0.021568


AUC исходного ID: 0.799514
AUC номера без префикса: 0.501958


## Утечка 2: record_id

Технический номер может быть известен уже при оформлении.
Подозрение возникает не из-за названия номера, а из-за того,
как формируется его диапазон в выгрузке.

В верхнем диапазоне доля опозданий около 26.19%, в нижнем — 2.16%.
Среди всех опоздавших заказов около 80% имеют номер из верхнего
диапазона; среди своевременных — около 19.91%.

Таким образом, диапазон номера сообщает модели информацию,
сильно связанную с исходом. Одна из гипотез: при формировании
выгрузки диапазон назначался с использованием результата доставки
или партии записей, связанной с результатом. Без описания или кода
выгрузки точный механизм присвоения номера установить нельзя.

Эта находка показывает, что нельзя доверять столбцам только по их
названию: нужно проверить происхождение и время расчёта остальных
полей. Я исключаю record_id из признаков модели.

In [8]:
reconstructed_score = (
    1 - df.groupby("partner_id")["late_delivery"].transform("mean")
).round(4)

matches = df["seller_score"].eq(reconstructed_score)

print("Доля совпадений:", matches.mean())
print("Количество несовпадений:", (~matches).sum())

orders_per_seller = (
    df.groupby("partner_id")["late_delivery"].transform("size")
)

single_order_sellers = df.loc[orders_per_seller == 1]

print("Продавцов с одним заказом:", len(single_order_sellers))

display(pd.crosstab(
    single_order_sellers["late_delivery"],
    single_order_sellers["seller_score"]
))

Доля совпадений: 1.0
Количество несовпадений: 0
Продавцов с одним заказом: 541


seller_score,0.0,1.0
late_delivery,,
0,0,488
1,53,0


In [9]:
id_is_high = df["record_id"] >= 500_000

display(
    pd.crosstab(
        df["late_delivery"],
        id_is_high,
        normalize="index",
    ).round(4)
)

record_id,False,True
late_delivery,,
0,0.8009,0.1991
1,0.2000,0.8000


## Утечка 3: seller_score

seller_score точно совпадает с долей своевременных доставок продавца,
рассчитанной по всей выгрузке и округлённой до четырёх знаков.

Агрегат включает исход текущего заказа и исходы заказов из test.
У продавца с единственным заказом score напрямую раскрывает его исход:
1 для своевременного заказа и 0 для опоздавшего.

Удаляем признак. 

Показатель продавца может быть полезным, поэтому в рабочей системе
его можно пересчитывать без утечки.

Для каждого заказа на момент его оформления нужно брать только
предыдущие заказы этого продавца, результат доставки которых к этому
моменту уже известен. Нельзя использовать исход текущего заказа,
будущие заказы или незавершённые заказы с ещё неизвестным результатом.

Это правило нужно соблюдать и для обучающих, и для проверочных строк.
Одного разделения по продавцам недостаточно: оно не исправляет значение,
которое уже рассчитано с использованием текущего таргета и будущих
исходов.

Если у продавца ещё нет истории, можно использовать заранее выбранное
значение по умолчанию, например общий показатель по ранее завершённым
заказам. При его расчёте тоже нельзя использовать будущие исходы.
В этом ДЗ я новый признак не создаю.

In [10]:
leak_columns = [
    "handover_lag_days",
    "record_id",
    "seller_score"
]

results = [{
    "variant": "Исходная выгрузка",
    "n_features": len(all_features),
    "roc_auc": baseline_auc
}]

for leak in leak_columns:
    columns = [
        column
        for column in all_features
        if column != leak
    ]

    auc = evaluate_model(columns)

    results.append({
        "variant": f"Без {leak}",
        "n_features": len(columns),
        "roc_auc": auc
    })

display(pd.DataFrame(results).round(6))

,variant,n_features,roc_auc
0,Исходная выгрузка,25,0.911209
1,Без handover_lag_days,24,0.891885
2,Без record_id,24,0.821679
3,Без seller_score,24,0.897571


In [11]:
clean_features = [
    column
    for column in all_features
    if column not in leak_columns
]

clean_auc = evaluate_model(clean_features)

all_results = results + [{
    "variant": "Без всех трёх утечек",
    "n_features": len(clean_features),
    "roc_auc": clean_auc
}]

results_table = pd.DataFrame(all_results)
results_table["drop_from_baseline"] = (
    baseline_auc - results_table["roc_auc"]
)

display(results_table.round(6))

,variant,n_features,roc_auc,drop_from_baseline
0,Исходная выгрузка,25,0.911209,0.000000
1,Без handover_lag_days,24,0.891885,0.019324
2,Без record_id,24,0.821679,0.089530
3,Без seller_score,24,0.897571,0.013638
4,Без всех трёх утечек,22,0.725824,0.185385


## Вывод

Исходная выгрузка дала ROC-AUC около 0.912. После удаления трёх утечек
он снизился примерно до 0.727 на том же разбиении и с той же моделью.
Это правильный результат: модель перестала получать информацию,
которой не будет при предсказании во время оформления заказа.

handover_lag_days раскрывал дальнейший ход доставки. record_id
содержал информацию об исходе через выбор диапазона технических номеров.
seller_score включал таргет текущей строки и других заказов из всей
выгрузки. При удалении только одного столбца две другие утечки оставались,
поэтому отдельные удаления не показывают качество полностью очищенных данных.
Их эффекты не обязаны складываться.

Последней в этом разборе проверен seller_score. Он выглядит как
полезная характеристика продавца, поэтому одного названия и AUC
недостаточно. Решающее доказательство - точное совпадение с агрегатом
таргета по всей выгрузке и пример продавцов с единственным заказом.

Чтобы проверить, нет ли других утечек, я бы сначала разобрал каждый
столбец: его смысл, источник, формулу расчёта и доступность в момент
оформления заказа. Затем проверил бы подозрительные поля по значениям
и сравнению моделей с признаками и без них.

Удаление признака не гарантирует обнаружение каждой утечки. Например,
если два признака содержат похожую информацию из будущего, удаление
только одного может почти не изменить качество: второй всё ещё
передаёт модели эту информацию.

Даже после таких проверок полностью исключить риск нельзя без
понимания исходной системы и способа формирования выгрузки.

In [12]:
for _, row in results_table.iterrows():
    print(f"| {row['variant']} | {row['roc_auc']:.6f} |")

| Исходная выгрузка | 0.911209 |
| Без handover_lag_days | 0.891885 |
| Без record_id | 0.821679 |
| Без seller_score | 0.897571 |
| Без всех трёх утечек | 0.725824 |
